# Hyperparameter Tuning

We will go through through the implementation of finding the best hyper parameters in a given neural network. Following are the hyper parameters which we are going to discuss:

   - Optimization algorithms
   - Activation function
   - Learning rate
   - Weights
   - Bias
   - Number of nodes in a layer
   - L0, L1 and L2 regularization
   - Number of epochs

For each of these, we will train the model for 100 epochs and provide the results in Tensor board for us to visualize. Once, we have the optimum value from each of these, then we will use them to improve our model to identify a Bollywood character's age group.

Most of the code in the above eight hyper parameters is same initially, which actually deals with loading necessary libraries, reading training data, converting it to a form suitable for the neural network, normalizing it and label encoding the output classes.

## Preprocessing

### Installing Packages

In [1]:
%pip install numpy pandas scikit-learn tensorflow keras imageio pillow

Note: you may need to restart the kernel to use updated packages.


### Importing libraries

In [2]:
import os
import re
import numpy as np
import pandas as pd
from keras.models import Sequential
from keras.layers import Dense, Flatten, InputLayer
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import utils
import keras
import imageio 
from PIL import Image
import tensorflow as tf

2025-06-15 15:54:41.347149: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-15 15:54:41.881918: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-15 15:54:42.055350: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750002882.605859    1613 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750002882.716518    1613 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750002885.296571    1613 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

### Reading the data

In [3]:
import zipfile

# Reading the data from zipped CSV files
with zipfile.ZipFile('datasets/agedetectiontrain.zip') as z:
	with z.open('train.csv') as f:
		train = pd.read_csv(f)

### Image resizing of train data into single numpy array

In [ ]:
temp = []
# Zipfile reading and image processing
with zipfile.ZipFile('datasets/agedetectiontrain.zip') as z:
    for img_name in train.ID:
        with z.open(f'Train/{img_name}') as img_file:
            img = imageio.imread(img_file)
            img = np.array(Image.fromarray(img).resize((32, 32))).astype('float32')
            temp.append(img)

train_x = np.stack(temp)

/tmp/ipykernel_1613/2118266606.py:6: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img = imageio.imread(img_file)


### Normalizing the images

In [5]:
train_x = train_x / 255.

### Encoding the categorical variable to numeric

In [6]:
lb = LabelEncoder()
train_y = lb.fit_transform(train.Class)
train_y = keras.utils.to_categorical(train_y)

### Specifying all the parameters we will be using in our network

In [7]:
input_num_units = (32, 32, 3)
hidden_num_units = 500
output_num_units = 3
epochs = 100
batch_size = 128

## Activation Functions

We are choosing 'linear', 'sigmoid', 'tanh', 'relu', and 'softmax' as the competing activation functions for the hidden layer.

### Building model functions usin activation functions now

Next, let us write a function to build a model for each optimizer and save the accuracy, loss, validation accuracy and validation loss for Tensor board visualization.

In [8]:
def models_with_different_activation_fn(list_of_activation_fn):    
    
    for i in range(len(list_of_activation_fn)):        
        # Defining the network
        model = Sequential([
          InputLayer(input_shape=input_num_units),
          Flatten(),
          Dense(units=hidden_num_units, activation=list_of_activation_fn[i]),
          Dense(units=output_num_units, activation='softmax'),
        ])
        model.compile(loss='categorical_crossentropy',
                  optimizer=keras.optimizers.Adam(),
                  metrics=['accuracy'])
        # Traning the model and writing log files for TensorBoard in distinct directories                
        logdir = f'optims2/{list_of_activation_fn[i]}' # Each log file needs to be written in a distinct directory. (Mandatory)
        
        # Writing graph will take time. Hence, keeping it False.
        cb = keras.callbacks.TensorBoard(log_dir=logdir, write_graph=False)         
        print('Building model using '+ list_of_activation_fn[i] + ' activation function')
        
        history = model.fit(train_x, train_y, epochs=epochs, 
                           validation_split=0.2,
                           callbacks=[cb])
        print('Model built sucessfully.')
        print('')

### Listing activation functions

In [9]:
act = ['linear', 'sigmoid', 'tanh', 'relu', 'softmax']

### Calling the function

In [10]:
models_with_different_activation_fn(act)

/home/codespace/.python/current/lib/python3.12/site-packages/keras/src/layers/core/input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(
2025-06-15 15:55:27.591911: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Building model using linear activation function
Epoch 1/100


2025-06-15 15:55:28.196539: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 195674112 exceeds 10% of free system memory.


496/498 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.5259 - loss: 1.9109

2025-06-15 15:55:35.517382: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 48930816 exceeds 10% of free system memory.


...

496/498 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.5483 - loss: 1.0177

2025-06-15 16:06:23.559158: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 48930816 exceeds 10% of free system memory.


498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.5485 - loss: 1.0170 - val_accuracy: 0.6253 - val_loss: 0.8189
Epoch 2/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.6145 - loss: 0.8302 - val_accuracy: 0.6178 - val_loss: 0.8124
Epoch 3/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.6267 - loss: 0.8120 - val_accuracy: 0.6502 - val_loss: 0.7684
Epoch 4/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.6439 - loss: 0.7901 - val_accuracy: 0.6354 - val_loss: 0.8098
Epoch 5/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.6474 - loss: 0.7859 - val_accuracy: 0.6351 - val_loss: 0.8069
Epoch 6/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 13ms/step - accuracy: 0.6445 - loss: 0.7814 - val_accuracy: 0.6665 - val_loss: 0.7649
Epoch 7/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6572 - loss: 0.7682 - val_accuracy: 0.6683 - val_loss: 0.7436
Epoch 8/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.6652 - loss: 0.7567 - val_accuracy

2025-06-15 16:16:03.542643: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 195674112 exceeds 10% of free system memory.


498/498 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.5301 - loss: 1.3946 - val_accuracy: 0.6223 - val_loss: 0.8313
Epoch 2/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6085 - loss: 0.8509 - val_accuracy: 0.6256 - val_loss: 0.8362
Epoch 3/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6099 - loss: 0.8550 - val_accuracy: 0.5844 - val_loss: 0.8560
Epoch 4/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6049 - loss: 0.8468 - val_accuracy: 0.6323 - val_loss: 0.8602
Epoch 5/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6126 - loss: 0.8339 - val_accuracy: 0.6238 - val_loss: 0.8094
Epoch 6/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6279 - loss: 0.8104 - val_accuracy: 0.6205 - val_loss: 0.8618
Epoch 7/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6273 - loss: 0.8188 - val_accuracy: 0.6328 - val_loss: 0.8290
Epoch 8/100
498/498 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.6244 - loss: 0.8142 - val_accuracy

This will result in a folder named “optims” which consists of the evaluation metrics. We can visualize the output by running the following command:

In [11]:
!tensorboard --logdir optims2

2025-06-15 16:41:16.440590: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-15 16:41:16.444825: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-15 16:41:16.460613: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750005676.477835   28730 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750005676.482266   28730 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750005676.495222   28730 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin